<a href="https://colab.research.google.com/github/Frogs-are-cool/Escape-room-game-thingy/blob/main/Escape_room_game_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
import json, os

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/escape_room_save"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_FILE = os.path.join(SAVE_DIR, "save.json")

Mounted at /content/drive


In [3]:
ROOMS = {
    "start": {
        "description": "You wake up on a cold stone floor. A locked door is to the North. "
                       "A dusty bookshelf stands against a wall.",
        "variants": [
            {"requires_item": "brass_key",
             "description": "The bookshelf has been searched, its books lying scattered. "
                            "The brass key in your hand fits the locked door to the North."}
        ],
        "choices": {
            "look at bookshelf": {"result": "A key falls out of a book! You pick it up.",
                                  "give_item": "brass_key", "next": "start"},
            "go north": {"result": "The key turns and the door swings open. You step through.",
                         "requires_item": "brass_key", "next": "hallway",
                         "fail_result": "The door won't budge. It's locked."},
        }
    },
    "hallway": {
        "description": "A narrow hallway. A riddle is carved into a wall: "
                       "'I speak without a mouth. What am I?'",
        "choices": {
            "answer echo": {"result": "A wall slides open, revealing a bright laboratory.",
                            "next": "lab"},
            "answer ghost": {"result": "Nothing happens. Wrong answer.", "next": "hallway"},
        }
    },
    "lab": {
        "description": "A bright, spotless laboratory. Four tables stand in a row, numbered 1 to 4. "
                       "In the middle of the room, a silver chalice sits on a stone pedestal, "
                       "engraved with a warning: 'THREE FAILED MIXTURES WILL BE YOUR LAST.' "
                       "A crumpled note on the floor reads: 'S,W,A,T'.",
        "variants": [
            {"requires_flag": "trapdoor_open",
             "description": "The silver chalice glows softly. In the floor, a trap door has slid "
                            "into view, an iron ring set into its lid. The note on the floor "
                            "still reads: 'S,W,A,T'."}
        ],
        "choices": {
            "inspect table 1": {"result": "On table 1 you find a jar of table salt. You take it.",
                                "give_item": "table_salt", "next": "lab"},
            "inspect table 2": {"result": "On table 2 you find a jug of water. You take it.",
                                "give_item": "water", "next": "lab"},
            "inspect table 3": {"result": "On table 3 you find a bag full of sand. You take it.",
                                "give_item": "sand", "next": "lab"},
            "inspect table 4": {"result": "On table 4 you find an apple juice box. You take it.",
                                "give_item": "apple_juice", "next": "lab"},
            "pour into chalice": {
                "puzzle": {
                    "id": "lab_chalice",
                    "answer": ["sand", "water", "apple_juice", "table_salt"],
                    "aliases": {
                        "sand": ["sand", "s"],
                        "water": ["water", "w"],
                        "apple_juice": ["apple juice", "juice", "apple", "a"],
                        "table_salt": ["table salt", "salt", "t"],
                    },
                    "max_attempts": 3,
                    "missing": "You stare at the chalice, but you don't have everything you need yet.",
                    "prompt": "In what order do you pour the items into the chalice? Separate them with commas.",
                    "failure": "The liquid fizzes, turns a murky brown, and drains away. "
                               "The chalice is empty again. Maybe the order matters.",
                    "lockout": "The mixture turns an angry red and begins to shake. The chalice "
                               "cracks, and the warning on its side comes true.",
                    "success": "The mixture swirls and glows gold. Stone grinds against stone as a "
                               "trap door slides into view in the floor.",
                    "done": "The chalice is already glowing. There's nothing more to pour.",
                    "sets_flag": "trapdoor_open",
                    "lose_room": "lab_explosion",
                },
            },
            "open trap door": {"result": "You heave the trap door open and climb down a ladder "
                                         "into a cold stone vault.",
                               "shown_if_flag": "trapdoor_open", "next": "vault"},
        }
    },
    "lab_explosion": {
        "description": "The chalice bursts in a blinding flash. The blast hurls you across the lab, "
                       "glass raining down around you as the lights go out. When the smoke clears, "
                       "the tables are splinters and the way back is buried under rubble.",
        "choices": {},  # empty choices marks an ending
        "ending": "lose"
    },
    "vault": {
        "description": "A cold stone vault. A heavy steel door fills the far wall, with a keypad "
                       "beside it that takes a two-digit number. A red label warns: 'THREE WRONG "
                       "CODES TRIGGER LOCKDOWN.' Chalked on the wall above the keypad: "
                       "'2, 4, 8, 16, ...'",
        "variants": [
            {"requires_flag": "vault_unlocked",
             "description": "The keypad glows green and the heavy steel door stands open, daylight "
                            "streaming through. The chalked sequence on the wall still reads: "
                            "'2, 4, 8, 16, ...'"}
        ],
        "choices": {
            "enter code": {
                "code_puzzle": {
                    "id": "vault_keypad",
                    "answer": "32",
                    "max_attempts": 3,
                    "prompt": "The keypad glows, waiting for a two-digit number.",
                    "success": "The keypad chimes, the lock clunks, and the steel door swings open.",
                    "failure": "BZZT. The keypad flashes red.",
                    "lockout": "BZZT. The keypad screams, and the label's warning comes true.",
                    "done": "The keypad is already glowing green. The door is open.",
                    "sets_flag": "vault_unlocked",
                    "lose_room": "lockdown",
                },
            },
            "open steel door": {"result": "You step through the open door into blinding daylight.",
                                "shown_if_flag": "vault_unlocked", "next": "final_room"},
        }
    },
    "lockdown": {
        "description": "Red lights flash and steel shutters slam over the ladder behind you. "
                       "Vents in the ceiling hiss, filling the vault with a sweet-smelling gas. "
                       "Your knees buckle and the floor rushes up to meet you. Nobody ever finds "
                       "the vault again.",
        "choices": {},  # empty choices marks an ending
        "ending": "lose"
    },
    "final_room": {
        "description": "Sunlight pours in from an open window. You've escaped!",
        "choices": {},  # empty choices marks an ending
        "ending": "win"
    }
}
STARTING_ROOM = "start"

In [4]:
def save_game(state):
  tmp = SAVE_FILE + ".tmp"
  with open(tmp, "w") as f:
    json.dump(state, f, indent=2)
  os.replace(tmp, SAVE_FILE)

def load_game():
  try:
    with open(SAVE_FILE) as f:
      return json.load(f)
  except (FileNotFoundError, json.JSONDecodeError):
    return None

def new_state():
  return {"version": 1, "current_room": STARTING_ROOM, "inventory": []}

In [5]:
def get_description(room, state):
  text = room["description"]
  for variant in room.get("variants", []):
    has_item = "requires_item" not in variant or variant["requires_item"] in state["inventory"]
    has_flag = "requires_flag" not in variant or variant["requires_flag"] in state.get("flags", [])
    if has_item and has_flag:
      text = variant["description"]
  return text

In [6]:
def run_puzzle(puzzle, state):
  flag = puzzle["sets_flag"]
  if flag in state["flags"]:
    print(puzzle["done"])
    return None

  missing = [item for item in puzzle["answer"] if item not in state["inventory"]]
  if missing:
    print(puzzle["missing"])
    return None

  lookup = {}
  for item, names in puzzle["aliases"].items():
    for name in names:
      lookup[name] = item

  print(puzzle["prompt"])
  raw = input("> ").strip().lower()
  parts = [p.strip() for p in raw.split(",") if p.strip()]

  if not parts:
    print("You decide not to pour anything yet. No attempt used.")
    return None

  order = []
  for part in parts:
    if part not in lookup:
      print(f"You're not sure what '{part}' means here, so you pour nothing. No attempt used.")
      return None
    order.append(lookup[part])

  if order == puzzle["answer"]:
    print(puzzle["success"])
    state["flags"].append(flag)
    return None

  attempts = state["attempts"].get(puzzle["id"], 0) + 1
  state["attempts"][puzzle["id"]] = attempts
  remaining = puzzle["max_attempts"] - attempts

  if remaining <= 0:
    print(puzzle["lockout"])
    return puzzle["lose_room"]

  print(puzzle["failure"])
  print(f"Attempts left: {remaining}")
  return None

In [7]:
def get_choices(state):
  room = ROOMS[state["current_room"]]
  return {k.lower(): v for k, v in room["choices"].items()
          if "shown_if_flag" not in v or v["shown_if_flag"] in state["flags"]}

def ending_of(state):
  """Returns 'win', 'lose', or None while the game is still going."""
  room = ROOMS[state["current_room"]]
  if room["choices"]:
    return None
  return room.get("ending", "lose")

def blocked(state, choice):
  """A message explaining why a choice can't be used right now, or None."""
  if "requires_item" in choice and choice["requires_item"] not in state["inventory"]:
    return choice.get("fail_result", "You can't do that yet.")
  for key in ("puzzle", "code_puzzle"):
    if key in choice:
      p = choice[key]
      if p["sets_flag"] in state["flags"]:
        return p["done"]
      if "aliases" in p and any(i not in state["inventory"] for i in p["answer"]):
        return p["missing"]
  return None

def do_choice(state, name):
  """Runs a plain choice (no typed input). Returns a list of messages."""
  choice = get_choices(state)[name]
  msgs = []
  if "give_item" in choice and choice["give_item"] in state["inventory"]:
    msgs.append("You already took that.")
  else:
    msgs.append(choice["result"])
    if "give_item" in choice:
      item = choice["give_item"]
      state["inventory"].append(item)
      msgs.append(f"You now have: {item.replace('_', ' ')}")
  state["current_room"] = choice["next"]
  return msgs

def register_miss(state, puzzle):
  attempts = state["attempts"].get(puzzle["id"], 0) + 1
  state["attempts"][puzzle["id"]] = attempts
  remaining = puzzle["max_attempts"] - attempts
  if remaining <= 0:
    return [puzzle["lockout"]], puzzle["lose_room"]
  return [puzzle["failure"], f"Attempts left: {remaining}"], None

def answer_order_puzzle(state, puzzle, text):
  """Returns (messages, lose_room or None)."""
  lookup = {}
  for item, names in puzzle["aliases"].items():
    for name in names:
      lookup[name] = item
  parts = [p.strip() for p in text.lower().split(",") if p.strip()]
  if not parts:
    return ["You decide not to pour anything yet. No attempt used."], None
  order = []
  for part in parts:
    if part not in lookup:
      return [f"You're not sure what '{part}' means here, so you pour nothing. No attempt used."], None
    order.append(lookup[part])
  if order == puzzle["answer"]:
    state["flags"].append(puzzle["sets_flag"])
    return [puzzle["success"]], None
  return register_miss(state, puzzle)

def answer_code_puzzle(state, puzzle, text):
  """Returns (messages, lose_room or None)."""
  guess = text.strip()
  if not guess.isdigit():
    return ["The keypad only accepts digits. No attempt used."], None
  if guess == puzzle["answer"]:
    state["flags"].append(puzzle["sets_flag"])
    return [puzzle["success"]], None
  return register_miss(state, puzzle)

In [8]:
import html
import ipywidgets as widgets
from IPython.display import display

def fresh_state():
  s = new_state()
  s.update(flags=[], attempts={})
  return s

loaded = load_game()
state = loaded if loaded else fresh_state()
state.setdefault("flags", [])      # lets older saves load without a KeyError
state.setdefault("attempts", {})

pending = {"kind": None, "puzzle": None}   # a puzzle waiting for typed input
confirm = {"armed": False}                 # for the two-click New Game button

inventory_html = widgets.HTML()
room_html = widgets.HTML()
result_html = widgets.HTML()
choice_box = widgets.HBox(layout=widgets.Layout(flex_flow="row wrap"))
prompt_html = widgets.HTML()
text_in = widgets.Text(placeholder="Type your answer here", layout=widgets.Layout(width="320px"))
submit_btn = widgets.Button(description="Submit", button_style="success")
cancel_btn = widgets.Button(description="Cancel")
prompt_box = widgets.VBox([prompt_html, widgets.HBox([text_in, submit_btn, cancel_btn])],
                          layout=widgets.Layout(display="none"))
save_btn = widgets.Button(description="Save", icon="save")
new_btn = widgets.Button(description="New Game", button_style="danger")

def show(msgs):
  result_html.value = "<br>".join(html.escape(m) for m in msgs)

def refresh(msgs=None):
  if msgs is not None:
    show(msgs)
  items = ", ".join(i.replace("_", " ") for i in state["inventory"]) or "nothing"
  inventory_html.value = f"<b>Inventory:</b> {html.escape(items)}"
  room = ROOMS[state["current_room"]]
  room_html.value = f"<p style='font-size:15px'>{html.escape(get_description(room, state))}</p>"
  prompt_box.layout.display = "none"
  choice_box.children = ()

  end = ending_of(state)
  save_btn.disabled = bool(end)
  if end:
    if os.path.exists(SAVE_FILE):
      os.remove(SAVE_FILE)   # a finished game shouldn't resume into its ending
    text = "You escaped! Congratulations." if end == "win" else "GAME OVER."
    result_html.value += f"<br><br><b>{text}</b> Press New Game to play again."
    return

  buttons = []
  for name in get_choices(state):
    b = widgets.Button(description=name)
    b.on_click(lambda btn, n=name: on_choice(n))
    buttons.append(b)
  choice_box.children = buttons

def on_choice(name):
  choice = get_choices(state)[name]
  reason = blocked(state, choice)
  if reason:
    show([reason])
    return
  for kind in ("puzzle", "code_puzzle"):
    if kind in choice:
      pending["kind"], pending["puzzle"] = kind, choice[kind]
      prompt_html.value = f"<i>{html.escape(choice[kind]['prompt'])}</i>"
      text_in.value = ""
      prompt_box.layout.display = ""
      for b in choice_box.children:
        b.disabled = True
      return
  msgs = do_choice(state, name)
  save_game(state)
  refresh(msgs)

def on_submit(btn):
  if pending["kind"] is None:
    return
  answer = answer_order_puzzle if pending["kind"] == "puzzle" else answer_code_puzzle
  msgs, lose_room = answer(state, pending["puzzle"], text_in.value)
  pending["kind"] = pending["puzzle"] = None
  if lose_room:
    state["current_room"] = lose_room
  save_game(state)
  refresh(msgs)

def on_cancel(btn):
  pending["kind"] = pending["puzzle"] = None
  refresh([])

def on_save(btn):
  save_game(state)
  show(["Game saved."])

def on_new(btn):
  if not confirm["armed"]:
    confirm["armed"] = True
    new_btn.description = "Click again to confirm"
    return
  confirm["armed"] = False
  new_btn.description = "New Game"
  if os.path.exists(SAVE_FILE):
    os.remove(SAVE_FILE)
  state.clear()
  state.update(fresh_state())
  pending["kind"] = pending["puzzle"] = None
  refresh(["You wake up in an unfamiliar room..."])

submit_btn.on_click(on_submit)
cancel_btn.on_click(on_cancel)
save_btn.on_click(on_save)
new_btn.on_click(on_new)

ui = widgets.VBox([inventory_html, room_html, result_html, choice_box, prompt_box,
                   widgets.HBox([save_btn, new_btn])])
refresh(["Welcome back! Your progress was restored."] if loaded
        else ["You wake up in an unfamiliar room..."])
display(ui)